# Notebook 03 — Customer RFM

## Brazilian E-Commerce Public Dataset by Olist

### Objetivo executivo

Transformar o histórico transacional em uma visão acionável de **valor, recorrência e risco de relacionamento** dos clientes. O notebook constrói uma dimensão RFM no nível de `customer_unique_id`, mede concentração de receita, separa clientes de compra única de recorrentes e traduz os resultados em segmentos úteis para CRM, retenção e alocação de investimento.

### Perguntas de negócio

1. Qual é o perfil de valor e recorrência da base de clientes?
2. Quantos clientes efetivamente recompram dentro do período observado?
3. Quanto da receita depende de clientes recorrentes e dos clientes de maior valor?
4. Quais segmentos RFM devem ser priorizados para retenção, expansão e reativação?
5. Como o meio principal de pagamento varia entre segmentos de clientes?
6. Quais outputs devem alimentar as análises de recompra, coorte e previsão?

> **Unidade de análise:** `customer_unique_id`. O campo `customer_id` identifica o registro associado a cada pedido e pode mudar entre compras do mesmo consumidor; por isso, não deve ser usado para medir recorrência.


## 1. Configuração e carregamento

O notebook utiliza `orders_base.csv`, produzido no Notebook 01 com uma linha por pedido e pagamentos/itens previamente agregados.

Essa arquitetura evita dupla contagem de receita ao juntar tabelas de granularidades diferentes.

A análise principal considera apenas pedidos:

- `order_status == "delivered"`;
- com `customer_unique_id` válido;
- com `order_purchase_timestamp` válida;
- com `payment_value` válido e não negativo para a métrica Monetary.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
orders_base_path = PROCESSED_DIR / "orders_base.csv"

if not orders_base_path.exists():
    raise FileNotFoundError(
        "orders_base.csv não encontrado. "
        "Execute o Notebook 01 antes do Notebook 03."
    )

orders_base = pd.read_csv(
    orders_base_path,
    parse_dates=["order_purchase_timestamp"]
)

required = {
    "order_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "payment_value",
}

missing = required.difference(orders_base.columns)

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes em orders_base: {sorted(missing)}"
    )

print(
    f"orders_base: {orders_base.shape[0]:,} linhas "
    f"x {orders_base.shape[1]} colunas"
)


In [ ]:
display(orders_base.head())

display(
    orders_base[
        [
            "order_id",
            "customer_unique_id",
            "order_status",
            "order_purchase_timestamp",
            "payment_value",
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing")
)


## 2. Universo analítico e data de referência

O RFM deve ser interpretado como um **snapshot**. A data de referência é definida como um dia após a última compra entregue observada.

Assim, Recency mede quantos dias se passaram desde a última compra até um ponto fixo e reproduzível.

### Regra financeira

`payment_value` representa o valor consolidado do pedido.

Pedidos entregues com `payment_value` ausente **não são silenciosamente convertidos para zero**. Eles são identificados e excluídos da construção de Monetary, pois não há evidência suficiente para afirmar que representam compras de valor zero.

Valores negativos também são excluídos como proteção de qualidade.


In [ ]:
rfm_candidate_orders = orders_base.loc[
    orders_base["order_status"].eq("delivered")
    & orders_base["customer_unique_id"].notna()
    & orders_base["order_purchase_timestamp"].notna()
].copy()

rfm_candidate_orders["payment_value"] = pd.to_numeric(
    rfm_candidate_orders["payment_value"],
    errors="coerce"
)

payment_quality = pd.Series({
    "delivered_candidate_orders": len(rfm_candidate_orders),
    "missing_payment_value":
        int(rfm_candidate_orders["payment_value"].isna().sum()),
    "negative_payment_value":
        int(rfm_candidate_orders["payment_value"].lt(0).sum()),
})

payment_quality


In [ ]:
rfm_orders = rfm_candidate_orders.loc[
    rfm_candidate_orders["payment_value"].notna()
    & rfm_candidate_orders["payment_value"].ge(0)
].copy()

reference_date = (
    rfm_orders["order_purchase_timestamp"].max().normalize()
    + pd.Timedelta(days=1)
)

coverage = pd.DataFrame({
    "metric": [
        "Pedidos candidatos entregues",
        "Pedidos válidos para RFM",
        "Pedidos excluídos por payment_value",
        "Clientes únicos",
        "Receita considerada",
        "Primeira compra",
        "Última compra",
        "Data de referência",
    ],
    "value": [
        len(rfm_candidate_orders),
        len(rfm_orders),
        len(rfm_candidate_orders) - len(rfm_orders),
        rfm_orders["customer_unique_id"].nunique(),
        rfm_orders["payment_value"].sum(),
        rfm_orders["order_purchase_timestamp"].min(),
        rfm_orders["order_purchase_timestamp"].max(),
        reference_date,
    ],
})

coverage


## 3. Construção das métricas RFM

Para cada consumidor:

- **Recency:** dias desde a última compra;
- **Frequency:** número de pedidos entregues válidos;
- **Monetary:** valor pago acumulado;
- **AOV:** valor médio por pedido;
- **Lifetime observado:** dias entre primeira e última compra;
- **Repeat customer:** indicador de 2 ou mais pedidos.

A separação explícita entre clientes de compra única e recorrentes é essencial. Em uma base muito concentrada em `frequency = 1`, forçar quintis de frequência pode criar diferenças artificiais entre clientes com comportamento idêntico.


In [ ]:
agg_dict = {
    "first_purchase": (
        "order_purchase_timestamp",
        "min",
    ),
    "last_purchase": (
        "order_purchase_timestamp",
        "max",
    ),
    "frequency": (
        "order_id",
        "nunique",
    ),
    "monetary": (
        "payment_value",
        "sum",
    ),
}

if "item_count" in rfm_orders.columns:
    agg_dict["total_items"] = (
        "item_count",
        lambda s: s.sum(min_count=1),
    )

rfm = (
    rfm_orders
    .groupby("customer_unique_id", as_index=False)
    .agg(**agg_dict)
)

rfm["recency"] = (
    reference_date
    - rfm["last_purchase"].dt.normalize()
).dt.days

rfm["customer_lifetime_days"] = (
    rfm["last_purchase"].dt.normalize()
    - rfm["first_purchase"].dt.normalize()
).dt.days

rfm["avg_order_value"] = (
    rfm["monetary"] / rfm["frequency"]
)

rfm["repeat_customer"] = (
    rfm["frequency"].ge(2)
)

cols = [
    "customer_unique_id",
    "first_purchase",
    "last_purchase",
    "recency",
    "frequency",
    "monetary",
    "avg_order_value",
    "customer_lifetime_days",
    "repeat_customer",
]

if "total_items" in rfm.columns:
    cols.insert(7, "total_items")

rfm = rfm[cols]

rfm.head()


In [ ]:
rfm_summary = (
    rfm[
        [
            "recency",
            "frequency",
            "monetary",
            "avg_order_value",
            "customer_lifetime_days",
        ]
    ]
    .describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
    .T
)

rfm_summary


## 4. Recorrência observada

Antes de pontuar RFM, medimos diretamente a distribuição de frequência.

Esse diagnóstico evita que o score esconda uma característica estrutural da base: a diferença entre aquisição pontual e relacionamento recorrente.


In [ ]:
frequency_distribution = (
    rfm["frequency"]
    .value_counts()
    .sort_index()
    .rename_axis("orders_per_customer")
    .reset_index(name="customers")
)

frequency_distribution["customer_share"] = (
    frequency_distribution["customers"]
    / len(rfm)
)

revenue_by_frequency = (
    rfm
    .groupby("frequency")["monetary"]
    .sum()
)

frequency_distribution["revenue"] = (
    frequency_distribution["orders_per_customer"]
    .map(revenue_by_frequency)
)

frequency_distribution["revenue_share"] = (
    frequency_distribution["revenue"]
    / rfm["monetary"].sum()
)

frequency_distribution.head(15)


In [ ]:
repeat_customers = int(
    rfm["repeat_customer"].sum()
)

total_customers = len(rfm)

repeat_rate = (
    repeat_customers / total_customers
)

repeat_revenue_share = (
    rfm.loc[
        rfm["repeat_customer"],
        "monetary"
    ].sum()
    / rfm["monetary"].sum()
)

repeat_kpis = pd.DataFrame({
    "kpi": [
        "customers",
        "repeat_customers_2plus",
        "observed_repeat_rate",
        "repeat_customer_revenue_share",
    ],
    "value": [
        total_customers,
        repeat_customers,
        repeat_rate,
        repeat_revenue_share,
    ],
})

repeat_kpis


In [ ]:
plot_freq = frequency_distribution.copy()

plot_freq["frequency_band"] = np.where(
    plot_freq["orders_per_customer"].ge(5),
    "5+",
    plot_freq["orders_per_customer"].astype(str),
)

plot_freq = (
    plot_freq
    .groupby("frequency_band", as_index=False)
    .agg(
        customers=("customers", "sum")
    )
)

frequency_order = [
    "1",
    "2",
    "3",
    "4",
    "5+",
]

plot_freq["frequency_band"] = pd.Categorical(
    plot_freq["frequency_band"],
    categories=frequency_order,
    ordered=True,
)

plot_freq = (
    plot_freq
    .sort_values("frequency_band")
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    plot_freq["frequency_band"].astype(str),
    plot_freq["customers"],
)

ax.set_title(
    "Distribuição de clientes por número de pedidos"
)
ax.set_xlabel("Pedidos entregues por cliente")
ax.set_ylabel("Clientes")

plt.tight_layout()
plt.show()


## 5. Distribuição de Recency, Frequency e Monetary

Frequency e Monetary tendem a ser assimétricos.

Os gráficos abaixo preservam todos os clientes nos cálculos e utilizam limites visuais apenas quando necessário.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(
    rfm["recency"],
    bins=40,
)
axes[0].set_title(
    "Recency — dias desde a última compra"
)
axes[0].set_xlabel("Dias")

axes[1].hist(
    rfm["frequency"],
    bins=np.arange(
        0.5,
        rfm["frequency"].max() + 1.5,
        1,
    ),
)
axes[1].set_xlim(
    0.5,
    min(
        rfm["frequency"].quantile(.995) + 1,
        10,
    ),
)
axes[1].set_title(
    "Frequency — pedidos por cliente"
)
axes[1].set_xlabel("Pedidos")

m99 = rfm["monetary"].quantile(.99)

axes[2].hist(
    rfm.loc[
        rfm["monetary"].le(m99),
        "monetary",
    ],
    bins=45,
)
axes[2].set_title(
    "Monetary — até P99 para visualização"
)
axes[2].set_xlabel("Valor acumulado")

plt.tight_layout()
plt.show()


## 6. Scores RFM — método adaptado à distribuição

### Recency e Monetary

Recebem notas de 1 a 5 com base na posição relativa do cliente na distribuição.

### Frequency

Não usamos `qcut` diretamente, porque clientes com o mesmo número de pedidos não devem receber notas distintas apenas para preencher quintis.

A regra é:

- 1 pedido → F=1
- 2 pedidos → F=2
- 3 pedidos → F=3
- 4 pedidos → F=4
- 5+ pedidos → F=5

Essa regra mantém o significado comportamental da recorrência e torna a segmentação mais interpretável para negócio.


In [ ]:
def quantile_score(
    series,
    higher_is_better=True,
):
    pct = (
        series
        .rank(
            method="average",
            pct=True,
        )
    )

    score = (
        np.ceil(pct * 5)
        .clip(1, 5)
        .astype(int)
    )

    if not higher_is_better:
        score = 6 - score

    return score


rfm["r_score"] = quantile_score(
    rfm["recency"],
    higher_is_better=False,
)

rfm["m_score"] = quantile_score(
    rfm["monetary"],
    higher_is_better=True,
)

rfm["f_score"] = (
    rfm["frequency"]
    .clip(upper=5)
    .astype(int)
)

rfm["rfm_score"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)

rfm["rfm_total_score"] = (
    rfm[
        [
            "r_score",
            "f_score",
            "m_score",
        ]
    ]
    .sum(axis=1)
)

score_diagnostics = pd.DataFrame({
    "r_score":
        rfm["r_score"]
        .value_counts()
        .sort_index(),
    "f_score":
        rfm["f_score"]
        .value_counts()
        .sort_index(),
    "m_score":
        rfm["m_score"]
        .value_counts()
        .sort_index(),
}).fillna(0).astype(int)

score_diagnostics


## 7. Segmentação executiva

Os segmentos são definidos por regras mutuamente priorizadas.

A lógica distingue explicitamente clientes de compra única de clientes recorrentes e dá precedência a clientes valiosos em risco.

| Segmento | Leitura principal |
|---|---|
| Champions | Recentes, recorrentes e de alto valor |
| Loyal Customers | Recorrência ativa e valor relevante |
| Potential Loyalists | Recorrência inicial e boa recência |
| New High Value | Uma compra recente, mas ticket/valor elevado |
| New Customers | Uma compra recente e valor ainda baixo/médio |
| Promising | Compra única relativamente recente com algum potencial de valor |
| Need Attention | Engajamento esfriando |
| Can't Lose Them | Histórico muito relevante, mas afastado |
| At Risk | Algum valor/recorrência histórica e baixa recência |
| Hibernating | Baixo sinal de valor e baixa recência |


In [ ]:
def assign_segment(row):
    r = row["r_score"]
    f = row["f_score"]
    m = row["m_score"]
    freq = row["frequency"]

    if r <= 2 and f >= 3 and m >= 4:
        return "Can't Lose Them"

    if r >= 4 and f >= 3 and m >= 4:
        return "Champions"

    if r >= 3 and f >= 2 and m >= 3:
        return "Loyal Customers"

    if r >= 4 and freq >= 2:
        return "Potential Loyalists"

    if r >= 4 and freq == 1 and m >= 4:
        return "New High Value"

    if r >= 4 and freq == 1:
        return "New Customers"

    if r == 3 and freq == 1 and m >= 3:
        return "Promising"

    if r == 3 and (
        freq >= 2 or m >= 3
    ):
        return "Need Attention"

    if r <= 2 and (
        freq >= 2 or m >= 4
    ):
        return "At Risk"

    return "Hibernating"


segment_order = [
    "Champions",
    "Loyal Customers",
    "Potential Loyalists",
    "New High Value",
    "New Customers",
    "Promising",
    "Need Attention",
    "Can't Lose Them",
    "At Risk",
    "Hibernating",
]

rfm["segment"] = (
    rfm.apply(
        assign_segment,
        axis=1,
    )
)

rfm["segment"] = pd.Categorical(
    rfm["segment"],
    categories=segment_order,
    ordered=True,
)

(
    rfm["segment"]
    .value_counts()
    .reindex(segment_order)
    .fillna(0)
    .astype(int)
)


## 8. Perfil dos segmentos

A tabela consolida tamanho, receita, recorrência e ticket.

Para decisão de investimento, dois eixos merecem atenção simultânea:

- participação no valor;
- tamanho da audiência.


In [ ]:
segment_summary = (
    rfm
    .groupby(
        "segment",
        observed=False,
    )
    .agg(
        customers=(
            "customer_unique_id",
            "nunique",
        ),
        revenue=(
            "monetary",
            "sum",
        ),
        avg_recency=(
            "recency",
            "mean",
        ),
        avg_frequency=(
            "frequency",
            "mean",
        ),
        repeat_rate=(
            "repeat_customer",
            "mean",
        ),
        avg_customer_value=(
            "monetary",
            "mean",
        ),
        median_customer_value=(
            "monetary",
            "median",
        ),
        avg_order_value=(
            "avg_order_value",
            "mean",
        ),
    )
    .reset_index()
)

segment_summary["customer_share"] = (
    segment_summary["customers"]
    / segment_summary["customers"].sum()
)

segment_summary["revenue_share"] = (
    segment_summary["revenue"]
    / segment_summary["revenue"].sum()
)

segment_summary = (
    segment_summary
    .sort_values(
        "revenue",
        ascending=False,
    )
)

segment_summary


In [ ]:
plot_df = (
    segment_summary
    .sort_values(
        "revenue",
        ascending=True,
    )
)

fig, ax = plt.subplots(
    figsize=(11, 6)
)

ax.barh(
    plot_df["segment"].astype(str),
    plot_df["revenue"],
)

ax.set_title(
    "Receita acumulada por segmento RFM"
)
ax.set_xlabel(
    "Valor pago acumulado"
)
ax.set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
segment_scatter = (
    segment_summary.loc[
        segment_summary["customers"].gt(0)
    ]
    .copy()
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

ax.scatter(
    segment_scatter["customer_share"],
    segment_scatter["revenue_share"],
    s=np.maximum(
        segment_scatter["avg_customer_value"],
        20,
    ),
    alpha=.65,
)

for _, row in segment_scatter.iterrows():
    ax.annotate(
        str(row["segment"]),
        (
            row["customer_share"],
            row["revenue_share"],
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_title(
    "Segmentos: participação de clientes × participação da receita"
)
ax.set_xlabel(
    "Participação dos clientes"
)
ax.set_ylabel(
    "Participação da receita"
)

plt.tight_layout()
plt.show()


## 9. Concentração de receita e Pareto

A concentração mostra quanto do valor histórico está concentrado em uma fração pequena dos consumidores.

Isso ajuda a dimensionar:

- oportunidade de fidelização;
- exposição econômica a clientes de maior valor;
- relevância de estratégias seletivas de retenção.


In [ ]:
rfm_ranked = (
    rfm
    .sort_values(
        "monetary",
        ascending=False,
    )
    .reset_index(drop=True)
)

rfm_ranked["customer_rank"] = (
    np.arange(
        1,
        len(rfm_ranked) + 1,
    )
)

rfm_ranked["customer_share_cum"] = (
    rfm_ranked["customer_rank"]
    / len(rfm_ranked)
)

rfm_ranked["revenue_cum"] = (
    rfm_ranked["monetary"]
    .cumsum()
)

rfm_ranked["revenue_share_cum"] = (
    rfm_ranked["revenue_cum"]
    / rfm_ranked["monetary"].sum()
)

concentration_rows = []

for pct in [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]:
    n = max(
        1,
        int(
            np.ceil(
                len(rfm_ranked) * pct
            )
        ),
    )

    concentration_rows.append({
        "top_customer_share": pct,
        "customers": n,
        "revenue_share":
            rfm_ranked.loc[
                n - 1,
                "revenue_share_cum",
            ],
    })

concentration = pd.DataFrame(
    concentration_rows
)

concentration


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    rfm_ranked["customer_share_cum"],
    rfm_ranked["revenue_share_cum"],
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
)

ax.set_title(
    "Curva de concentração da receita por cliente"
)
ax.set_xlabel(
    "Participação acumulada dos clientes"
)
ax.set_ylabel(
    "Participação acumulada da receita"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 10. Matriz Recency × Monetary

Como a maior parte dos consumidores possui baixa frequência, a matriz **R × M** é especialmente útil para separar clientes de alto valor ainda recentes daqueles de alto valor que já esfriaram.


In [ ]:
rm_matrix = (
    rfm
    .groupby(
        ["r_score", "m_score"],
        as_index=False,
    )
    .agg(
        customers=(
            "customer_unique_id",
            "nunique",
        ),
        revenue=(
            "monetary",
            "sum",
        ),
    )
)

rm_customers = (
    rm_matrix
    .pivot(
        index="r_score",
        columns="m_score",
        values="customers",
    )
    .reindex(
        index=[5, 4, 3, 2, 1],
        columns=[1, 2, 3, 4, 5],
    )
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

mat = (
    rm_customers
    .fillna(0)
    .to_numpy()
)

im = ax.imshow(
    mat,
    aspect="auto",
)

ax.set_xticks(
    range(
        len(rm_customers.columns)
    ),
    rm_customers.columns,
)

ax.set_yticks(
    range(
        len(rm_customers.index)
    ),
    rm_customers.index,
)

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(
            j,
            i,
            f"{mat[i, j]:.0f}",
            ha="center",
            va="center",
        )

fig.colorbar(
    im,
    ax=ax,
    label="Clientes",
)

ax.set_title(
    "Clientes por score de Recency × Monetary"
)
ax.set_xlabel(
    "M Score — valor"
)
ax.set_ylabel(
    "R Score — recência"
)

plt.tight_layout()
plt.show()

rm_customers


## 11. Matriz Frequency × Monetary

Essa matriz evidencia a diferença entre clientes de **alto ticket pontual** e clientes cujo valor foi construído por **recorrência**.


In [ ]:
fm_matrix = (
    rfm
    .groupby(
        ["f_score", "m_score"],
        as_index=False,
    )
    .agg(
        customers=(
            "customer_unique_id",
            "nunique",
        ),
        revenue=(
            "monetary",
            "sum",
        ),
        avg_monetary=(
            "monetary",
            "mean",
        ),
    )
)

fm_customers = (
    fm_matrix
    .pivot(
        index="f_score",
        columns="m_score",
        values="customers",
    )
    .reindex(
        index=[5, 4, 3, 2, 1],
        columns=[1, 2, 3, 4, 5],
    )
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

mat = (
    fm_customers
    .fillna(0)
    .to_numpy()
)

im = ax.imshow(
    mat,
    aspect="auto",
)

ax.set_xticks(
    range(
        len(fm_customers.columns)
    ),
    fm_customers.columns,
)

ax.set_yticks(
    range(
        len(fm_customers.index)
    ),
    fm_customers.index,
)

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(
            j,
            i,
            f"{mat[i, j]:.0f}",
            ha="center",
            va="center",
        )

fig.colorbar(
    im,
    ax=ax,
    label="Clientes",
)

ax.set_title(
    "Clientes por score de Frequency × Monetary"
)
ax.set_xlabel(
    "M Score — valor"
)
ax.set_ylabel(
    "F Score — frequência"
)

plt.tight_layout()
plt.show()

fm_customers


## 12. Top clientes e clientes valiosos em risco

Um ranking baseado apenas em Monetary mistura clientes ainda ativos com clientes possivelmente perdidos.

Por isso, são produzidas duas listas:

- **top value**;
- **high-value at risk**.


In [ ]:
top_customers = (
    rfm
    .sort_values(
        [
            "monetary",
            "frequency",
        ],
        ascending=[
            False,
            False,
        ],
    )
    [
        [
            "customer_unique_id",
            "segment",
            "recency",
            "frequency",
            "monetary",
            "avg_order_value",
            "last_purchase",
            "rfm_score",
            "rfm_total_score",
        ]
    ]
    .head(20)
)

top_customers


In [ ]:
high_value_at_risk = (
    rfm.loc[
        rfm["segment"]
        .astype(str)
        .isin(
            [
                "Can't Lose Them",
                "At Risk",
            ]
        )
    ]
    .sort_values(
        [
            "monetary",
            "frequency",
        ],
        ascending=[
            False,
            False,
        ],
    )
    [
        [
            "customer_unique_id",
            "segment",
            "recency",
            "frequency",
            "monetary",
            "avg_order_value",
            "last_purchase",
        ]
    ]
    .head(50)
)

high_value_at_risk.head(20)


## 13. RFM × meio principal de pagamento

Se `payment_behavior_order_level.csv` existir, o notebook cruza o **meio principal do pedido**, produzido pelo Notebook 02, com o segmento RFM.

Como cada pedido possui apenas um `primary_payment_type`, a composição dentro de cada segmento é mutuamente exclusiva e soma 100%.

Essa leitura é descritiva e representa associação, não causalidade.


In [ ]:
payment_behavior_path = (
    PROCESSED_DIR
    / "payment_behavior_order_level.csv"
)

payment_segment_mix = pd.DataFrame()

if payment_behavior_path.exists():
    payment_behavior = pd.read_csv(
        payment_behavior_path
    )

    expected_payment_cols = {
        "order_id",
        "primary_payment_type",
    }

    if expected_payment_cols.issubset(
        payment_behavior.columns
    ):
        order_customer_segment = (
            rfm_orders[
                [
                    "order_id",
                    "customer_unique_id",
                ]
            ]
            .merge(
                rfm[
                    [
                        "customer_unique_id",
                        "segment",
                    ]
                ],
                on="customer_unique_id",
                how="inner",
                validate="many_to_one",
            )
        )

        pay_seg = (
            payment_behavior[
                [
                    "order_id",
                    "primary_payment_type",
                ]
            ]
            .merge(
                order_customer_segment,
                on="order_id",
                how="inner",
                validate="one_to_one",
            )
        )

        payment_segment_mix = (
            pay_seg
            .groupby(
                [
                    "segment",
                    "primary_payment_type",
                ],
                observed=False,
            )
            .agg(
                orders=(
                    "order_id",
                    "nunique",
                )
            )
            .reset_index()
        )

        payment_segment_mix[
            "segment_orders"
        ] = (
            payment_segment_mix
            .groupby(
                "segment",
                observed=False,
            )["orders"]
            .transform("sum")
        )

        payment_segment_mix[
            "primary_payment_share"
        ] = (
            payment_segment_mix["orders"]
            / payment_segment_mix[
                "segment_orders"
            ]
        )

        display(
            payment_segment_mix
            .sort_values(
                [
                    "segment",
                    "orders",
                ],
                ascending=[
                    True,
                    False,
                ],
            )
        )

    else:
        print(
            "Arquivo do Notebook 02 encontrado, "
            "mas sem as colunas order_id/"
            "primary_payment_type esperadas."
        )

else:
    print(
        "Output do Notebook 02 não encontrado; "
        "cruzamento de pagamentos será pulado."
    )


In [ ]:
if not payment_segment_mix.empty:
    pay_pivot = (
        payment_segment_mix
        .pivot(
            index="segment",
            columns="primary_payment_type",
            values="primary_payment_share",
        )
        .fillna(0)
    )

    display(pay_pivot)

    ax = pay_pivot.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 6),
    )

    ax.set_title(
        "Mix do meio principal de pagamento por segmento RFM"
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        "Participação dos pedidos"
    )

    plt.xticks(
        rotation=45,
        ha="right",
    )

    plt.tight_layout()
    plt.show()


## 14. Recomendações por segmento

A segmentação transforma o histórico em uma lógica de priorização comercial.

| Segmento | Objetivo | Ação sugerida |
|---|---|---|
| Champions | Defender valor | Benefícios, cross-sell, early access, referrals |
| Loyal Customers | Expandir share of wallet | Recomendação personalizada e bundles |
| Potential Loyalists | Consolidar hábito | Incentivo à 3ª compra e comunicação pós-compra |
| New High Value | Converter ticket em recorrência | Onboarding premium e segunda compra rápida |
| New Customers | Ativar recompra | Jornada de onboarding e incentivo moderado |
| Promising | Desenvolver valor | Personalização por categoria e timing |
| Need Attention | Recuperar engajamento | Reativação leve e mensagem contextual |
| Can't Lose Them | Evitar perda de valor | Win-back prioritário e oferta seletiva |
| At Risk | Recuperar clientes historicamente bons | Testes de win-back com limite de CAC |
| Hibernating | Controlar custo | Comunicação barata; evitar subsídio indiscriminado |

### Implicação para investidores

Uma taxa de recorrência baixa aumenta a dependência de aquisição contínua.

O RFM não prova causalidade, mas identifica onde existe maior valor histórico e onde testes de retenção podem ser priorizados.


## 15. KPIs executivos

Os indicadores abaixo são exportados em formato simples para reutilização no relatório, na apresentação e no Power BI.


In [ ]:
total_revenue = (
    rfm["monetary"].sum()
)

champions_revenue = (
    rfm.loc[
        rfm["segment"]
        .astype(str)
        .eq("Champions"),
        "monetary",
    ]
    .sum()
)

risk_revenue = (
    rfm.loc[
        rfm["segment"]
        .astype(str)
        .isin(
            [
                "Can't Lose Them",
                "At Risk",
            ]
        ),
        "monetary",
    ]
    .sum()
)

top10_share = (
    concentration.loc[
        np.isclose(
            concentration[
                "top_customer_share"
            ],
            .10,
        ),
        "revenue_share",
    ]
    .iloc[0]
)

top20_share = (
    concentration.loc[
        np.isclose(
            concentration[
                "top_customer_share"
            ],
            .20,
        ),
        "revenue_share",
    ]
    .iloc[0]
)

kpis = pd.DataFrame({
    "kpi": [
        "unique_customers",
        "total_revenue",
        "avg_customer_value",
        "median_customer_value",
        "repeat_customers_2plus",
        "observed_repeat_rate",
        "repeat_customer_revenue_share",
        "champions_revenue_share",
        "at_risk_plus_cant_lose_revenue_share",
        "top_10pct_customer_revenue_share",
        "top_20pct_customer_revenue_share",
    ],
    "value": [
        len(rfm),
        total_revenue,
        rfm["monetary"].mean(),
        rfm["monetary"].median(),
        repeat_customers,
        repeat_rate,
        repeat_revenue_share,
        (
            champions_revenue
            / total_revenue
            if total_revenue
            else np.nan
        ),
        (
            risk_revenue
            / total_revenue
            if total_revenue
            else np.nan
        ),
        top10_share,
        top20_share,
    ],
})

kpis


## 16. Sanity checks e governança

Os checks abaixo devem falhar explicitamente se houver quebra de granularidade ou inconsistência lógica.


In [ ]:
assert rfm[
    "customer_unique_id"
].is_unique, (
    "customer_rfm deve ter uma linha "
    "por customer_unique_id."
)

assert rfm[
    "frequency"
].ge(1).all(), (
    "Frequency inválida."
)

assert rfm[
    "recency"
].ge(0).all(), (
    "Recency negativa."
)

assert rfm[
    "monetary"
].notna().all(), (
    "Monetary não deve ser nulo "
    "na dimensão final."
)

assert rfm[
    "monetary"
].ge(0).all(), (
    "Monetary negativo."
)

assert rfm[
    "r_score"
].between(1, 5).all()

assert rfm[
    "f_score"
].between(1, 5).all()

assert rfm[
    "m_score"
].between(1, 5).all()

assert np.isclose(
    segment_summary["revenue"].sum(),
    rfm["monetary"].sum(),
)

assert np.isclose(
    segment_summary["customers"].sum(),
    len(rfm),
)

if not payment_segment_mix.empty:
    segment_payment_sums = (
        payment_segment_mix
        .groupby(
            "segment",
            observed=False,
        )["primary_payment_share"]
        .sum()
    )

    non_empty = segment_payment_sums.gt(0)

    assert np.allclose(
        segment_payment_sums.loc[
            non_empty
        ],
        1.0,
    ), (
        "O mix de meio principal por segmento "
        "deveria somar 100%."
    )

print(
    "Sanity checks concluídos com sucesso."
)


## 17. Exportação das camadas analíticas

Outputs principais:

- `customer_rfm.csv`: dimensão de cliente com métricas, scores e segmento;
- `rfm_segment_summary.csv`: visão executiva por segmento;
- `rfm_frequency_distribution.csv`: distribuição de pedidos por cliente;
- `rfm_revenue_concentration.csv`: Pareto/concentração;
- `rfm_top_customers.csv`: maiores clientes por valor;
- `rfm_high_value_at_risk.csv`: clientes valiosos em risco;
- `rfm_executive_kpis.csv`: KPIs para relatório, apresentação e Power BI;
- `rfm_payment_segment_mix.csv`: meio principal de pagamento por segmento, quando disponível.


In [ ]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

exports = {
    "customer_rfm.csv":
        rfm,
    "rfm_segment_summary.csv":
        segment_summary,
    "rfm_frequency_distribution.csv":
        frequency_distribution,
    "rfm_revenue_concentration.csv":
        concentration,
    "rfm_top_customers.csv":
        top_customers,
    "rfm_high_value_at_risk.csv":
        high_value_at_risk,
    "rfm_executive_kpis.csv":
        kpis,
    "rfm_rm_matrix.csv":
        rm_matrix,
    "rfm_fm_matrix.csv":
        fm_matrix,
}

if not payment_segment_mix.empty:
    exports[
        "rfm_payment_segment_mix.csv"
    ] = payment_segment_mix

for filename, df in exports.items():
    path = (
        PROCESSED_DIR
        / filename
    )

    df.to_csv(
        path,
        index=False,
    )

    print(
        "Salvo:",
        path,
    )


## 18. Checklist de interpretação

### O que este notebook mede bem

- valor histórico no nível do consumidor (`customer_unique_id`);
- recorrência observada dentro da janela do dataset;
- concentração de receita;
- recência e risco comportamental;
- segmentos de clientes para priorização comercial;
- associação entre segmento e meio principal de pagamento.

### Limitações

1. **RFM é descritivo e relativo ao período observado.** Não estima causalmente a probabilidade de recompra.
2. **Monetary é receita paga, não margem.** Não há CAC, custo de produto ou custo promocional no dataset.
3. **Clientes adquiridos perto do fim da janela têm menos tempo para recomprar.** A análise de coorte é necessária para comparar retenção de forma justa.
4. **Recorrência observada não é retenção ajustada por exposição.** Isso será tratado nos Notebooks 04 e 05.
5. **Segmentos são regras de negócio.** Devem ser validados com testes de campanha e comparados com modelos de propensão.
6. **Pedidos sem `payment_value` válido são excluídos da construção de Monetary.** Essa decisão evita interpretar ausência de informação como receita zero.

### Próxima etapa

O Notebook 04 deve decompor a recompra por tempo desde a primeira compra, categoria, ticket e pagamento.

O Notebook 05 deve construir coortes mensais para separar efeito de aquisição recente de retenção real.
